[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/weaviate-features/search-rest-api/search-rest-api.ipynb)

# The Weaviate Search REST API

*Weaviate version `1.40.0` · Weaviate Python client `4.23.1` · Cohere `embed-multilingual-v3.0` — by Duda Nogueira ([Weaviate](https://weaviate.io))*

Weaviate has a **dedicated REST API for search**: one `POST` endpoint per search type, with a JSON body in and a consistent JSON envelope out. No GraphQL and no gRPC are needed.

| Endpoint | Search type |
|---|---|
| `POST /v1/search/{collection}/bm25` | Keyword (BM25F) |
| `POST /v1/search/{collection}/near-text` | Vector search from text, vectorized by the collection's vectorizer |
| `POST /v1/search/{collection}/near-vector` | Vector search from a vector you provide |
| `POST /v1/search/{collection}/near-object` | Vector search from an existing object |
| `POST /v1/search/{collection}/hybrid` | Keyword + vector, fused into one ranking |
| `POST /v1/aggregate/{collection}` | Object counts, in total or per group |

`near-text` arrived in v1.39. **v1.40** completes the set and turns the API **on by default**.

This makes Weaviate search easy to call from places where a client library is a poor fit: `curl` and shell scripts, edge functions, low-code tools, or languages without an official client.

In this recipe we load Wikipedia passages that are **already vectorized with Cohere**, then call every endpoint with plain `requests`.

In [1]:
!pip install -Uq "weaviate-client>=4.23.1" requests datasets

## 1. Set your Cohere API key

The data comes with its vectors, so importing it costs nothing. Weaviate only calls Cohere to vectorize your **queries** for `near-text` and `hybrid`. You need a [Cohere API key](https://dashboard.cohere.com/api-keys); a free trial key works.

In [2]:
import os
from getpass import getpass

if not os.environ.get("COHERE_APIKEY"):
    os.environ["COHERE_APIKEY"] = getpass("Cohere API key: ")

## 2. Start an embedded Weaviate

We use [Embedded Weaviate](https://docs.weaviate.io/deploy/installation-guides/embedded) so this notebook runs anywhere, including Colab. To use another Weaviate instance, replace `connect_to_embedded()` with `connect_to_local()` or `connect_to_weaviate_cloud()` and update `BASE_URL`.

In [3]:
import weaviate

client = weaviate.connect_to_embedded(
    version="1.40.0",
    headers={"X-Cohere-Api-Key": os.environ["COHERE_APIKEY"]},
    environment_variables={"LOG_LEVEL": "error"},
)

BASE_URL = "http://localhost:8079/v1"  # Embedded Weaviate listens on port 8079

print("Weaviate version:", client.get_meta()["version"])

Weaviate version: 1.40.0


## 3. Create collections and load data

We use the Python client only for setup. Every search below is plain HTTP.

- `WikiPage` holds one object per Wikipedia page (`title`, `url`).
- `Passage` holds the text passages, with a `fromPage` cross-reference to their page so we can show `returnReferences` later. It uses the `text2vec-cohere` vectorizer with the **same model** that produced the dataset's vectors, so query vectors and stored vectors are comparable.

In [4]:
from weaviate.classes.config import Configure, DataType, Property, ReferenceProperty

client.collections.delete(["Passage", "WikiPage"])

pages = client.collections.create(
    "WikiPage",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="url", data_type=DataType.TEXT),
    ],
    vector_config=Configure.Vectors.self_provided(),
)

passages = client.collections.create(
    "Passage",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="text", data_type=DataType.TEXT),
    ],
    references=[ReferenceProperty(name="fromPage", target_collection="WikiPage")],
    vector_config=Configure.Vectors.text2vec_cohere(model="embed-multilingual-v3.0"),
)

We stream 2,000 passages from [`weaviate/wiki-sample`](https://huggingface.co/datasets/weaviate/wiki-sample), using its `cohere-embed-multilingual-v3` configuration. Each row carries its 1024-dimension vector, which we pass to Weaviate as is.

> The same dataset is also available with OpenAI vectors (`openai-text-embedding-3-small` and `openai-text-embedding-3-large`). To use those, switch the dataset configuration, use `Configure.Vectors.text2vec_openai(model=...)` and send an `X-OpenAI-Api-Key` header instead.

In [5]:
from itertools import islice
from datasets import load_dataset
from weaviate.util import generate_uuid5

stream = load_dataset("weaviate/wiki-sample", "cohere-embed-multilingual-v3", split="train", streaming=True)
rows = list(islice(stream, 2000))
print({k: (f"<{len(v)} floats>" if k == "vector" else v) for k, v in rows[0].items()})

{'title': 'Unicode', 'text': "The Unicode Standard includes more than just the base code. Alongside the character encodings, the Consortium's official publication includes a wide variety of details about the scripts and how to display them: normalization rules, decomposition, collation, rendering, and bidirectional text display order for multilingual texts, and so on.", 'wiki_id': '20231101.simple_64846_4', 'url': 'https://simple.wikipedia.org/wiki/Unicode', 'vector': '<1024 floats>'}


In [6]:
with pages.batch.fixed_size(500) as batch:
    for url, title in {r["url"]: r["title"] for r in rows}.items():
        batch.add_object({"title": title, "url": url}, uuid=generate_uuid5(url))

with passages.batch.fixed_size(500) as batch:
    for r in rows:
        batch.add_object(
            {"title": r["title"], "text": r["text"]},
            vector=r["vector"],  # pre-computed Cohere embedding: no vectorization at import
            references={"fromPage": generate_uuid5(r["url"])},
        )

print("Failed:", len(pages.batch.failed_objects) + len(passages.batch.failed_objects))
print("Pages:", pages.aggregate.over_all().total_count, "| Passages:", passages.aggregate.over_all().total_count)

Failed: 0
Pages: 1939 | Passages: 2000


## 4. A small helper

Every search endpoint returns the same envelope:

```json
{
  "results": [
    {"id": "...", "properties": {...}, "references": {...}, "metadata": {...}}
  ],
  "tookMs": 3
}
```

Requests that vectorize a query need the provider key. Over REST you send it as a header, the same `X-Cohere-Api-Key` header the client uses.

In [7]:
import json
import requests

http = requests.Session()
http.headers.update({"X-Cohere-Api-Key": os.environ["COHERE_APIKEY"]})


def search(endpoint, body, collection="Passage"):
    response = http.post(f"{BASE_URL}/search/{collection}/{endpoint}", json=body)
    response.raise_for_status()
    return response.json()


def show(result, field="title"):
    for obj in result["results"]:
        meta = obj.get("metadata", {})
        meta_str = " ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in meta.items())
        print(f"- {obj['properties'].get(field)}  [{meta_str}]")
    print(f"({len(result['results'])} results in {result['tookMs']} ms)")

## 5. Keyword search: `bm25`

In [8]:
result = search("bm25", {
    "query": "telescope",
    "queryProperties": ["title^2", "text"],  # boost matches in the title
    "limit": 3,
    "returnMetadata": ["score"],
})
show(result)

- Telescope  [score=12.770]
- William Herschel  [score=3.543]
(2 results in 2 ms)


Here is one raw result, so you can see the full envelope:

In [9]:
print(json.dumps(result["results"][0], indent=2))

{
  "id": "6dcaf591-d0b3-4fce-95ab-4120500eff94",
  "metadata": {
    "score": 12.770418
  },
  "properties": {
    "text": "A telescope (from the Ancient Greek \u03c4\u1fc6\u03bb\u03b5, tele \"far\" and \u03c3\u03ba\u03bf\u03c0\u03b5\u1fd6\u03bd, skopein \"to look or see\"; \u03c4\u03b7\u03bb\u03b5\u03c3\u03ba\u03cc\u03c0\u03bf\u03c2, teleskopos) makes astronomical objects appear closer to the naked eye. It is an important tool for astronomy that gathers light and directs it to a single point. Some do this with curved mirrors, some with curved lenses, and some with both. Telescopes make distant things look bigger, brighter and closer. Galileo was the first person to use a telescope for astronomy, but he did not invent them. The first telescope was invented in the Netherlands in 1608. Some telescopes, not mainly used for astronomy, are binoculars, camera lenses, or spyglasses.",
    "title": "Telescope"
  }
}


## 6. Vector search from text: `near-text`

Weaviate vectorizes the query with Cohere, then searches the stored vectors. `query` is a **list** of concepts.

In [10]:
result = search("near-text", {
    "query": ["huge explosions in deep space"],
    "limit": 3,
    "returnMetadata": ["distance"],
})
show(result)

- Stellar evolution  [distance=0.425]
- Emission nebula  [distance=0.553]
- Orion complex  [distance=0.566]
(3 results in 846 ms)


`embed-multilingual-v3.0` is multilingual, so the same search works in another language, here Portuguese:

In [11]:
show(search("near-text", {
    "query": ["grandes explosões no espaço profundo"],
    "limit": 3,
    "returnMetadata": ["distance"],
}))

- Stellar evolution  [distance=0.509]
- Orion complex  [distance=0.601]
- Accretion disk  [distance=0.602]
(3 results in 448 ms)


## 7. Vector search from a vector: `near-vector`

Use this when you compute the query embedding yourself. No provider call is made, so this endpoint doesn't need the API key. Here we reuse the stored vector of the first passage.

In [12]:
source = rows[0]
print("Source:", source["title"])

result = search("near-vector", {
    "vector": source["vector"],
    "limit": 3,
    "returnMetadata": ["distance"],
})
show(result)

Source: Unicode
- Unicode  [distance=0]
- Emoticon  [distance=0.469]
- Web design  [distance=0.483]
(3 results in 1 ms)


## 8. Vector search from an object: `near-object`

The same search, but Weaviate looks up the stored vector of the object `id` for you.

In [13]:
source_id = result["results"][0]["id"]

show(search("near-object", {
    "id": source_id,
    "limit": 3,
    "returnMetadata": ["distance"],
}))

- Unicode  [distance=0]
- Emoticon  [distance=0.469]
- Web design  [distance=0.483]
(3 results in 1 ms)


## 9. Hybrid search: `hybrid`

Combines a BM25 ranking and a vector ranking. `alpha` sets the weight of the vector side: `0` is pure keyword, `1` is pure vector.

In [14]:
for alpha in [0.0, 0.5, 1.0]:
    print(f"alpha={alpha}")
    show(search("hybrid", {
        "query": "rock bands from the United States",
        "alpha": alpha,
        "limit": 3,
        "returnMetadata": ["score"],
    }))
    print()

alpha=0.0
- Rock music  [score=1]
- Solicitor General of the United States  [score=0.672]
- First Amendment to the United States Constitution  [score=0.557]
(3 results in 1 ms)

alpha=0.5


- Rock music  [score=0.880]
- The Go-Go's  [score=0.600]
- Big Black  [score=0.465]
(3 results in 241 ms)

alpha=1.0


- The Go-Go's  [score=1]
- Big Black  [score=0.909]
- Rock music  [score=0.760]
(3 results in 259 ms)



## 10. Filters, returned fields and references

Every search endpoint shares the same common options:

- `where`: a filter, with the same shape as the [REST `where` filter](https://docs.weaviate.io/weaviate/api/rest)
- `returnProperties`: which properties to return
- `returnReferences`: which cross-references to follow, and what to return from them
- `returnMetadata`: `distance`, `certainty`, `score`, `explainScore`, `creationTime`, `lastUpdateTime`
- `limit`, `offset`, `autoLimit`, `tenant`, `consistencyLevel`

In [15]:
result = search("near-text", {
    "query": ["stars and galaxies"],
    "where": {"path": ["title"], "operator": "Like", "valueText": "List of*"},
    "returnProperties": ["title"],
    "returnReferences": [
        {"linkOn": "fromPage", "returnProperties": ["url"]},
    ],
    "returnMetadata": ["distance"],
    "limit": 2,
})
print(json.dumps(result, indent=2))

{
  "results": [
    {
      "id": "56a47a60-65a9-4497-9dca-0eb487f78eb8",
      "metadata": {
        "distance": 0.4400186
      },
      "properties": {
        "title": "List of most luminous stars"
      },
      "references": {
        "fromPage": [
          {
            "properties": {
              "url": "https://simple.wikipedia.org/wiki/List%20of%20most%20luminous%20stars"
            }
          }
        ]
      }
    },
    {
      "id": "92ac56bb-e08a-400e-82a6-21809193b2c4",
      "metadata": {
        "distance": 0.6451777
      },
      "properties": {
        "title": "List of World Heritage Sites in Northern and Central Asia"
      },
      "references": {
        "fromPage": [
          {
            "properties": {
              "url": "https://simple.wikipedia.org/wiki/List%20of%20World%20Heritage%20Sites%20in%20Northern%20and%20Central%20Asia"
            }
          }
        ]
      }
    }
  ],
  "tookMs": 230
}


## 11. Counting objects: `aggregate`

`POST /v1/aggregate/{collection}` returns object counts. An empty body returns the total count. Add `where` to count only matching objects, and `groupBy` to count per value of a property.

In [16]:
def aggregate(body, collection="Passage"):
    response = http.post(f"{BASE_URL}/aggregate/{collection}", json=body)
    response.raise_for_status()
    return response.json()


print("All passages:", aggregate({}))
print("'List of ...' passages:", aggregate({"where": {"path": ["title"], "operator": "Like", "valueText": "List of*"}}))

All passages: {'count': 2000, 'tookMs': 0}
'List of ...' passages: {'count': 124, 'tookMs': 2}


In [17]:
by_page = aggregate({"groupBy": "title"})
print("One group:", by_page["groups"][0], "\n")

for group in sorted(by_page["groups"], key=lambda g: -g["count"])[:5]:
    print(f"{group['count']:3}  {group['groupedBy']['value']}")

One group: {'count': 8, 'groupedBy': {'path': ['title'], 'value': 'List of minor planets: 4001–5000'}} 

  8  List of minor planets: 4001–5000
  4  List of minor planets: 6001–7000
  3  Deaths in September 2021
  3  List of minor planets: 3001–4000
  3  List of minor planets: 10001–11000


## 12. The same calls with `curl`

Because these are plain HTTP calls, you can run them from any shell:

In [18]:
!curl -s -X POST http://localhost:8079/v1/search/Passage/hybrid \
  -H 'Content-Type: application/json' \
  -H "X-Cohere-Api-Key: $COHERE_APIKEY" \
  -d '{"query": "olympic games", "limit": 2, "returnProperties": ["title"]}'

{"results":[{"id":"b2141ade-2ad5-4c99-b661-232fd31c254c","properties":{"title":"1994 Asian Games"}},{"id":"94328189-66b6-4f0b-9496-891796b57217","properties":{"title":"Athens Olympic Aquatic Centre"}}],"tookMs":223}


## REST or a client library?

The search REST API and the client libraries query the same engine and return the same objects. For comparison, here is the hybrid search from above with the Python client, which uses gRPC:

In [19]:
response = passages.query.hybrid("rock bands from the United States", alpha=0.5, limit=3)
for o in response.objects:
    print("-", o.properties["title"])

- Rock music
- The Go-Go's
- Big Black


- Use a **client library** for applications in a supported language: you get typed results, gRPC performance, batching and the full feature set.
- Use the **search REST API** when you only need HTTP: scripts, `curl`, serverless functions, low-code tools, or a language without a client.

Learn more in the [Weaviate REST API reference](https://docs.weaviate.io/weaviate/api/rest).

In [20]:
client.collections.delete(["Passage", "WikiPage"])
client.close()